In [ ]:
EPOCHS = 15000
LR = 1E-4
BATCH_SIZE = 1

In [10]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")

Using MPS (Apple Silicon)
Device: mps


In [ ]:
from radiance_field_project.data.datamodule import UCO3DDataModule

dm = UCO3DDataModule(
    batch_size=1,
    num_workers=0,
    image_height=512,
    image_width=512
)

torch.manual_seed(42)
train_loader = dm.train_loader()

print(f"Dataset size: {len(train_loader.dataset)}")
print(f"Num batches:  {len(train_loader)}")

Dataset size: 9399
Num batches:  9399


In [ ]:
from radiance_field_project.models.simple_autoencoder import SimpleAutoencoder

network = SimpleAutoencoder().to(device)


In [ ]:
import torch
from torch.optim import AdamW
import torch.nn.functional as F

from radiance_field_project.data.gaussian_features import batch_to_features

optimizer = AdamW(network.parameters(), lr=LR)

sample = next(iter(train_loader))

features, batchidx = batch_to_features(sample)
features = features.to(device)

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    recon = network(features)
    loss = F.mse_loss(features, recon)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Current loss = {loss.item():.6f}")


In [ ]:
from uco3d import render_splats
from radiance_field_project.data.gaussian_features import features_to_gaussians

from radiance_field_project.utils.splat_visualizer import SplatRenderVisualizer


render_colors_gt, render_alphas_gt, render_info_gt = render_splats(
    cameras=sample.camera,
    splats=sample.sequence_gaussian_splats,
    render_size=[512, 512]
)

SplatRenderVisualizer.show(render_colors=render_colors_gt, render_alphas=render_alphas_gt)

In [ ]:
network.eval()

with torch.no_grad():
    recon = network(features)

sequence_gaussians_splats_reconstructed = features_to_gaussians(recon)

render_colors_recon, render_alphas_recon, render_info_recon = render_splats(
    cameras=sample.camera,
    splats=sequence_gaussians_splats_reconstructed,
    render_size=[512, 512]
)

SplatRenderVisualizer.show(render_colors=render_colors_recon, render_alphas=render_alphas_recon)